# LeetCode 1302 — Deepest Leaves Sum bằng Python

Link đề: https://leetcode.com/problems/deepest-leaves-sum/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def deepestLeavesSum(self, root: TreeNode | None) -> int:
        ...
```

## 1. Mục tiêu học

1. Biết tự dựng cây nhị phân từ mảng **level-order** (có `None`) để test bài cây trên máy.
2. Hiểu khái niệm **độ sâu (depth)**, **tầng (level)**, **lá (leaf)** và nhận ra: nút ở tầng sâu nhất **luôn là lá**.
3. Giải bằng **BFS theo tầng** (dùng `deque`, xử lý trọn từng tầng) — đáp án là tổng tầng cuối cùng.
4. Giải bằng **DFS mang theo độ sâu** một lượt, với quy tắc "gặp tầng sâu hơn thì **reset** tổng".
5. Tránh lỗi: cộng **mọi** lá, quên reset tổng, tràn đệ quy với cây lệch sâu `10^4`.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân, trả về **tổng giá trị các lá ở tầng sâu nhất**.

**Ví dụ 1:** `root = [1,2,3,4,5,null,6,7,null,null,null,null,8]` → `15`

```
          1
        /   \
       2     3
      / \     \
     4   5     6
    /           \
   7             8
```
Tầng sâu nhất (độ sâu 3) gồm `7` và `8` ⇒ `7 + 8 = 15`.

**Ví dụ 2:** `root = [6,7,8,2,7,1,3,9,null,1,4,null,null,null,5]` → `19` (tầng sâu nhất: `9, 1, 4, 5`).

**Ràng buộc:** số nút trong `[1, 10^4]`, `1 <= Node.val <= 100`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def to_list(root):
    """Ngược lại: cây -> mảng level-order (bỏ None thừa ở cuối)."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([1, 2, 3, 4, 5, None, 6, 7, None, None, None, None, 8], 15),
    ([6, 7, 8, 2, 7, 1, 3, 9, None, 1, 4, None, None, None, 5], 19),
]
for vals, want in examples:
    root = build_tree(vals)
    print("dựng lại:", to_list(root), "| mong đợi", want)

dựng lại: [1, 2, 3, 4, 5, None, 6, 7, None, None, None, None, 8] | mong đợi 15
dựng lại: [6, 7, 8, 2, 7, 1, 3, 9, None, 1, 4, None, None, None, 5] | mong đợi 19


## 3. Kiến thức nền

### 3.1 Lớp `TreeNode` và mảng level-order

LeetCode biểu diễn cây bằng mảng duyệt **theo tầng, trái sang phải**; `null` (Python: `None`) là vị trí trống. Lưu ý: con của nút `None` **không** được ghi ra — nên không dùng công thức `2i+1, 2i+2` mà phải dựng bằng hàng đợi như `build_tree` ở trên.

Hàm vẽ cây nằm ngang dưới đây giúp nhìn cấu trúc (cây xoay 90°, gốc bên trái, con phải ở trên).

In [2]:
def show(root, indent=0):
    if root is None:
        return
    show(root.right, indent + 4)
    print(" " * indent + str(root.val))
    show(root.left, indent + 4)

show(build_tree(examples[0][0]))

            8
        6
    3
1
        5
    2
        4
            7


### 3.2 Độ sâu, tầng, lá

- **Độ sâu** của nút = số cạnh từ gốc tới nút (gốc có độ sâu 0). Các nút cùng độ sâu tạo thành một **tầng**.
- **Lá** = nút không có con nào.
- **Chiều cao cây** = độ sâu lớn nhất.

Nhận xét: nếu nút `x` ở tầng sâu nhất mà có con, con đó sẽ sâu hơn — mâu thuẫn. ⇒ **mọi nút ở tầng sâu nhất đều là lá**. Vậy bài toán = "tổng các nút ở tầng sâu nhất".

In [3]:
def is_leaf(node):
    return node is not None and node.left is None and node.right is None

def height(node):
    if node is None:
        return -1
    return 1 + max(height(node.left), height(node.right))

r = build_tree(examples[0][0])
print("chiều cao =", height(r), "| gốc là lá?", is_leaf(r), "| nút 7 là lá?", is_leaf(r.left.left.left))

chiều cao = 3 | gốc là lá? False | nút 7 là lá? True


### 3.3 `collections.deque` — hàng đợi hai đầu

`append` thêm vào cuối, `popleft` lấy ở đầu, cả hai `O(1)` (khác `list.pop(0)` là `O(n)`). Đây là cấu trúc chuẩn cho BFS. Mẹo **xử lý trọn một tầng**: tại đầu mỗi vòng, `len(q)` chính là số nút của tầng hiện tại.

In [4]:
q = deque([1, 2, 3])
q.append(4)
print(q.popleft(), q)          # 1 deque([2, 3, 4])

root = build_tree([1, 2, 3, 4, 5])
q = deque([root])
level = 0
while q:
    size = len(q)
    vals = []
    for _ in range(size):
        node = q.popleft()
        vals.append(node.val)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    print("tầng", level, "->", vals)
    level += 1

1 deque([2, 3, 4])
tầng 0 -> [1]
tầng 1 -> [2, 3]
tầng 2 -> [4, 5]


### 3.4 Đệ quy, `nonlocal` và giới hạn đệ quy

DFS đệ quy rất tự nhiên với cây. Để cập nhật biến của hàm ngoài từ hàm lồng, dùng `nonlocal`. Python mặc định giới hạn độ sâu đệ quy khoảng 1000 (`sys.getrecursionlimit()`); cây lệch `10^4` nút sẽ vượt quá ⇒ cần `sys.setrecursionlimit` hoặc viết bằng **stack tường minh**.

In [5]:
def count_nodes(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        total += 1
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return total

print("số nút ví dụ 2:", count_nodes(build_tree(examples[1][0])))
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

số nút ví dụ 2: 11
giới hạn đệ quy hiện tại: 1000


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2**: `[6,7,8,2,7,1,3,9,null,1,4,null,null,null,5]`

```
              6
          /       \
         7         8
       /   \      /  \
      2     7    1    3
     /     / \         \
    9     1   4         5
```

Đi từng tầng:
- Tầng 0: `6` → tổng 6
- Tầng 1: `7, 8` → tổng 15
- Tầng 2: `2, 7, 1, 3` → tổng 13
- Tầng 3: `9, 1, 4, 5` → tổng **19** ← tầng cuối cùng

Vậy chỉ cần **tổng của tầng cuối**. Chú ý lá `1` (con trái của `8`) ở tầng 2 **không** được tính vì không ở tầng sâu nhất.

In [6]:
def level_sums(root):
    sums, q = [], deque([root])
    while q:
        s = 0
        for _ in range(len(q)):
            node = q.popleft()
            s += node.val
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        sums.append(s)
    return sums

for vals, want in examples:
    ls = level_sums(build_tree(vals))
    print("tổng từng tầng:", ls, "-> tầng cuối =", ls[-1], "| mong đợi", want)

tổng từng tầng: [1, 5, 15, 15] -> tầng cuối = 15 | mong đợi 15
tổng từng tầng: [6, 15, 13, 19] -> tầng cuối = 19 | mong đợi 19


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** liệt kê **mọi đường đi từ gốc tới lá** (lưu cả danh sách nút trên đường). Độ sâu của lá = độ dài đường − 1. Tìm độ sâu lớn nhất, rồi cộng giá trị các lá có độ sâu đó.

Đây là cách "nghĩ thẳng" nhưng lãng phí: mỗi đường đi được sao chép thành list riêng.

In [7]:
def deepest_brute(root):
    paths = []
    def dfs(node, path):
        if node is None:
            return
        path = path + [node.val]               # sao chép đường đi (tốn kém)
        if node.left is None and node.right is None:
            paths.append(path)
            return
        dfs(node.left, path)
        dfs(node.right, path)
    dfs(root, [])
    max_len = max(len(p) for p in paths)
    return sum(p[-1] for p in paths if len(p) == max_len)

for vals, want in examples:
    print(deepest_brute(build_tree(vals)), "| mong đợi", want)

15 | mong đợi 15
19 | mong đợi 19


**Độ phức tạp:** gọi `n` = số nút, `h` = chiều cao.
- Time `O(n · h)`: mỗi lần `path + [val]` tốn `O(độ dài)`; tổng chi phí cỡ `Σ độ sâu` ≤ `n·h`, cây lệch ⇒ `O(n²)` = `10^8` với `n = 10^4`.
- Space `O(n · h)` để lưu mọi đường đi (cây lệch: một đường dài `n`; cây cân bằng: `n/2` đường dài `log n`).

**Nút thắt:** ta chỉ cần **độ sâu** và **giá trị lá**, không cần lưu cả đường đi.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** chỉ cần **độ sâu** của mỗi nút, không cần đường đi ⇒ truyền `depth` như một số nguyên.

**Insight 2 (hai lượt — gợi ý của đề):** lượt 1 tìm chiều cao `H`; lượt 2 cộng các nút có `depth == H`.

**Insight 3 (BFS):** duyệt theo tầng, mỗi tầng tính tổng; tổng của **tầng cuối cùng** là đáp án.

**Insight 4 (DFS một lượt):** giữ `max_depth` và `total`:
- **Nếu** `depth > max_depth` **thì** phát hiện tầng mới sâu hơn ⇒ `max_depth = depth`, **reset** `total = node.val`.
- **Nếu** `depth == max_depth` **thì** `total += node.val`.
- **Nếu** `depth < max_depth` **thì** bỏ qua.

**Invariant DFS:** sau khi đã thăm một tập nút, `max_depth` là độ sâu lớn nhất trong tập đó và `total` là tổng các nút đã thăm có độ sâu bằng `max_depth`.

In [8]:
# Minh hoạ invariant: thứ tự DFS và (max_depth, total) sau mỗi nút
root = build_tree(examples[0][0])
max_d, total = -1, 0
stack = [(root, 0)]
while stack:
    node, d = stack.pop()
    if d > max_d:
        max_d, total = d, node.val
    elif d == max_d:
        total += node.val
    print(f"thăm {node.val} (độ sâu {d}) -> max_depth={max_d}, total={total}")
    if node.right: stack.append((node.right, d + 1))
    if node.left: stack.append((node.left, d + 1))

thăm 1 (độ sâu 0) -> max_depth=0, total=1
thăm 2 (độ sâu 1) -> max_depth=1, total=2
thăm 4 (độ sâu 2) -> max_depth=2, total=4
thăm 7 (độ sâu 3) -> max_depth=3, total=7
thăm 5 (độ sâu 2) -> max_depth=3, total=7
thăm 3 (độ sâu 1) -> max_depth=3, total=7
thăm 6 (độ sâu 2) -> max_depth=3, total=7
thăm 8 (độ sâu 3) -> max_depth=3, total=15


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai lượt: tìm chiều cao rồi cộng

Đúng như gợi ý của đề. Dễ hiểu, `O(n)` nhưng duyệt cây hai lần.

In [9]:
def deepest_two_pass(root):
    def height(node):
        if node is None:
            return -1
        return 1 + max(height(node.left), height(node.right))
    H = height(root)
    def sum_at(node, d):
        if node is None:
            return 0
        if d == H:
            return node.val
        return sum_at(node.left, d + 1) + sum_at(node.right, d + 1)
    return sum_at(root, 0)

for vals, want in examples:
    print(deepest_two_pass(build_tree(vals)), "| mong đợi", want)

15 | mong đợi 15
19 | mong đợi 19


Code nộp LeetCode (cách 2, có nâng giới hạn đệ quy cho cây lệch):

In [10]:
class SolutionTwoPass:
    def deepestLeavesSum(self, root: Optional[TreeNode]) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))
        def height(node):
            return -1 if node is None else 1 + max(height(node.left), height(node.right))
        H = height(root)
        def sum_at(node, d):
            if node is None:
                return 0
            if d == H:
                return node.val
            return sum_at(node.left, d + 1) + sum_at(node.right, d + 1)
        return sum_at(root, 0)

print(SolutionTwoPass().deepestLeavesSum(build_tree(examples[1][0])))

19


**Độ phức tạp cách 2:** Time `O(n)` (2 lượt), Space `O(h)` cho stack đệ quy.

### 7.2 Cách 3 — BFS theo tầng

Duyệt từng tầng bằng `deque`. Ở đầu mỗi tầng đặt `level_sum = 0`, cộng mọi nút trong tầng. Khi hàng đợi rỗng, `level_sum` cuối cùng chính là tổng tầng sâu nhất. Không đệ quy ⇒ không lo giới hạn đệ quy.

In [11]:
def deepest_bfs(root):
    q = deque([root])
    level_sum = 0
    while q:
        level_sum = 0                      # reset cho tầng mới
        for _ in range(len(q)):
            node = q.popleft()
            level_sum += node.val
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    return level_sum

for vals, want in examples:
    print(deepest_bfs(build_tree(vals)), "| mong đợi", want)

15 | mong đợi 15
19 | mong đợi 19


Code nộp LeetCode (cách 3):

In [12]:
class SolutionBFS:
    def deepestLeavesSum(self, root: Optional[TreeNode]) -> int:
        q = deque([root])
        level_sum = 0
        while q:
            level_sum = 0
            for _ in range(len(q)):
                node = q.popleft()
                level_sum += node.val
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
        return level_sum

print(SolutionBFS().deepestLeavesSum(build_tree(examples[0][0])))

15


**Độ phức tạp cách 3:** Time `O(n)`, Space `O(w)` với `w` = độ rộng lớn nhất của một tầng (cây đầy đủ: `w ≈ n/2`).

Biến thể gọn hơn: thay vì đếm `len(q)`, mỗi vòng dựng **list tầng kế tiếp**:

In [13]:
def deepest_bfs_lists(root):
    level = [root]
    while True:
        nxt = [c for node in level for c in (node.left, node.right) if c]
        if not nxt:
            return sum(node.val for node in level)
        level = nxt

for vals, want in examples:
    print(deepest_bfs_lists(build_tree(vals)), "| mong đợi", want)

15 | mong đợi 15
19 | mong đợi 19


### 7.3 Cách 4 — DFS đệ quy một lượt mang theo độ sâu

Áp dụng Insight 4. Dùng `nonlocal` để cập nhật `max_depth`, `total`.

In [14]:
def deepest_dfs(root):
    max_depth, total = -1, 0
    def dfs(node, d):
        nonlocal max_depth, total
        if node is None:
            return
        if d > max_depth:
            max_depth, total = d, node.val     # tầng mới sâu hơn -> reset
        elif d == max_depth:
            total += node.val
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    dfs(root, 0)
    return total

for vals, want in examples:
    print(deepest_dfs(build_tree(vals)), "| mong đợi", want)

15 | mong đợi 15
19 | mong đợi 19


Code nộp LeetCode (cách 4):

In [15]:
class SolutionDFS:
    def deepestLeavesSum(self, root: Optional[TreeNode]) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))
        self.max_depth, self.total = -1, 0
        def dfs(node, d):
            if node is None:
                return
            if d > self.max_depth:
                self.max_depth, self.total = d, node.val
            elif d == self.max_depth:
                self.total += node.val
            dfs(node.left, d + 1)
            dfs(node.right, d + 1)
        dfs(root, 0)
        return self.total

print(SolutionDFS().deepestLeavesSum(build_tree(examples[1][0])))

19


**Độ phức tạp cách 4:** Time `O(n)` một lượt, Space `O(h)` stack đệ quy (cây lệch: `O(n)`).

### 7.4 Cách 5 — DFS bằng stack tường minh

Giống cách 4 nhưng tự quản lý stack các cặp `(node, depth)` ⇒ không bị giới hạn đệ quy, an toàn với cây lệch `10^4`.

In [16]:
def deepest_dfs_iter(root):
    max_depth, total = -1, 0
    stack = [(root, 0)]
    while stack:
        node, d = stack.pop()
        if d > max_depth:
            max_depth, total = d, node.val
        elif d == max_depth:
            total += node.val
        if node.left:
            stack.append((node.left, d + 1))
        if node.right:
            stack.append((node.right, d + 1))
    return total

for vals, want in examples:
    print(deepest_dfs_iter(build_tree(vals)), "| mong đợi", want)

15 | mong đợi 15
19 | mong đợi 19


**Độ phức tạp cách 5:** Time `O(n)`, Space `O(h)` cho stack (trong trường hợp xấu `O(n)`).

**Chọn cách nào?** BFS (cách 3) là tự nhiên nhất vì đề nói "tầng sâu nhất"; DFS một lượt (cách 4/5) tiết kiệm bộ nhớ với cây rộng.

## 8. TRACE từng bước

Trace BFS (cách 3) trên Ví dụ 1:

| Tầng | Hàng đợi đầu tầng | level_sum | Hàng đợi cuối tầng |
|---|---|---|---|
| 0 | [1] | 1 | [2, 3] |
| 1 | [2, 3] | 5 | [4, 5, 6] |
| 2 | [4, 5, 6] | 15 | [7, 8] |
| 3 | [7, 8] | **15** | [] → dừng |

Đáp án: `15` (tình cờ tầng 2 cũng có tổng 15, nhưng đáp án là tầng 3 = `7 + 8`).

In [17]:
def trace_bfs(root):
    q = deque([root])
    level, level_sum = 0, 0
    while q:
        start = [n.val for n in q]
        level_sum = 0
        for _ in range(len(q)):
            node = q.popleft()
            level_sum += node.val
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        print(f"tầng {level}: đầu tầng {start} -> level_sum={level_sum} | cuối tầng {[n.val for n in q]}")
        level += 1
    print("=> đáp án:", level_sum)
    return level_sum

trace_bfs(build_tree(examples[0][0]))

tầng 0: đầu tầng [1] -> level_sum=1 | cuối tầng [2, 3]
tầng 1: đầu tầng [2, 3] -> level_sum=5 | cuối tầng [4, 5, 6]
tầng 2: đầu tầng [4, 5, 6] -> level_sum=15 | cuối tầng [7, 8]
tầng 3: đầu tầng [7, 8] -> level_sum=15 | cuối tầng []
=> đáp án: 15


15

Trace DFS một lượt (cách 4) trên Ví dụ 2 — chú ý các lần **reset**:

In [18]:
def trace_dfs(root):
    state = {"max_depth": -1, "total": 0}
    def dfs(node, d):
        if node is None:
            return
        if d > state["max_depth"]:
            state["max_depth"], state["total"] = d, node.val
            act = "RESET (tầng mới sâu hơn)"
        elif d == state["max_depth"]:
            state["total"] += node.val
            act = "cộng thêm"
        else:
            act = "bỏ qua (nông hơn)"
        print(f"{'  ' * d}nút {node.val} d={d}: {act:26s} max_depth={state['max_depth']} total={state['total']}")
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    dfs(root, 0)
    print("=> đáp án:", state["total"])
    return state["total"]

trace_dfs(build_tree(examples[1][0]))

nút 6 d=0: RESET (tầng mới sâu hơn)   max_depth=0 total=6
  nút 7 d=1: RESET (tầng mới sâu hơn)   max_depth=1 total=7
    nút 2 d=2: RESET (tầng mới sâu hơn)   max_depth=2 total=2
      nút 9 d=3: RESET (tầng mới sâu hơn)   max_depth=3 total=9
    nút 7 d=2: bỏ qua (nông hơn)          max_depth=3 total=9
      nút 1 d=3: cộng thêm                  max_depth=3 total=10
      nút 4 d=3: cộng thêm                  max_depth=3 total=14
  nút 8 d=1: bỏ qua (nông hơn)          max_depth=3 total=14
    nút 1 d=2: bỏ qua (nông hơn)          max_depth=3 total=14
    nút 3 d=2: bỏ qua (nông hơn)          max_depth=3 total=14
      nút 5 d=3: cộng thêm                  max_depth=3 total=19
=> đáp án: 19


19

## 9. Code nộp LeetCode

Bản chính thức: BFS theo tầng — không đệ quy, chạy tốt cả với cây lệch `10^4` nút. (Trên LeetCode lớp `TreeNode` đã có sẵn; ở đây ta dùng lớp tự dựng ở mục 2.)

In [19]:
class Solution:
    def deepestLeavesSum(self, root: TreeNode | None) -> int:
        q = deque([root])
        level_sum = 0
        while q:
            level_sum = 0                     # tổng của tầng hiện tại
            for _ in range(len(q)):           # xử lý trọn một tầng
                node = q.popleft()
                level_sum += node.val
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
        return level_sum                      # tầng cuối cùng = sâu nhất

sol = Solution()
for vals, want in examples:
    got = sol.deepestLeavesSum(build_tree(vals))
    print(got, "✔" if got == want else "✘")

15 ✔
19 ✔


## 10. Vì sao thuật toán đúng

**BFS:** bất biến — ở đầu vòng `while` thứ `k`, hàng đợi chứa **đúng** các nút có độ sâu `k`, theo thứ tự trái sang phải.
- Đúng với `k = 0` (chỉ có gốc).
- Giả sử đúng với `k`: vòng `for` lấy ra đúng `len(q)` nút tầng `k` và đẩy vào toàn bộ con của chúng — chính là mọi nút tầng `k+1` (mỗi nút tầng `k+1` có đúng một cha ở tầng `k`). ⇒ đúng với `k+1`.

Vòng lặp dừng khi tầng `H+1` rỗng; khi đó `level_sum` là tổng tầng `H` = tầng sâu nhất, và như mục 3.2 đã chỉ ra, các nút này đều là lá ⇒ đúng đáp án.

**DFS một lượt:** chứng minh invariant ở mục 6 bằng quy nạp theo số nút đã thăm. Khi thăm nút mới độ sâu `d`: nếu `d > max_depth` thì tập "nút sâu nhất" mới chỉ gồm nút này ⇒ reset đúng; nếu bằng thì thêm vào; nếu nhỏ hơn thì không đổi. Sau khi thăm hết, invariant cho đúng đáp án.

## 11. Phân tích Time / Space

Gọi `n` = số nút (≤ `10^4`), `h` = chiều cao, `w` = độ rộng tầng lớn nhất.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê đường đi | `O(n·h)` | `O(n·h)` | cây lệch ⇒ `O(n²)` |
| 2. Hai lượt | `O(n)` | `O(h)` | duyệt 2 lần |
| 3. BFS | `O(n)` | `O(w)` | cây đầy đủ `w ≈ n/2` |
| 4. DFS đệ quy | `O(n)` | `O(h)` | cần nâng recursion limit |
| 5. DFS stack | `O(n)` | `O(h)` | an toàn với cây lệch |

- Cây **cân bằng**: `h ≈ log n`, `w ≈ n/2` ⇒ DFS tốn ít bộ nhớ hơn BFS.
- Cây **lệch** (dạng danh sách): `h = n − 1`, `w = 1` ⇒ BFS tốn ít bộ nhớ hơn DFS.

## 12. Bộ test

Ví dụ đề, edge case (1 nút, cây lệch trái/phải, cây đầy đủ, nhiều lá ở các tầng khác nhau), cây lệch rất sâu `10^4` nút, và đối chiếu mọi cách trên cây ngẫu nhiên.

In [20]:
ALL = [deepest_brute, deepest_two_pass, deepest_bfs, deepest_bfs_lists, deepest_dfs, deepest_dfs_iter,
       Solution().deepestLeavesSum, SolutionBFS().deepestLeavesSum,
       SolutionTwoPass().deepestLeavesSum, SolutionDFS().deepestLeavesSum]
tests = [
    (examples[0][0], 15),
    (examples[1][0], 19),
    ([5], 5),                                   # 1 nút
    ([1, 2, None, 3, None, 4], 4),              # lệch trái
    ([1, None, 2, None, 3], 3),                 # lệch phải
    ([1, 2, 3, 4, 5, 6, 7], 22),                # cây đầy đủ: 4+5+6+7
    ([1, 2, 3, None, None, 4, 5], 9),           # lá 2 nông hơn, không tính
    ([1, 2, 3, 4], 4),
]
for vals, want in tests:
    for f in ALL:
        got = f(build_tree(vals))
        assert got == want, (f, vals, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 8 test cố định qua cho 10 hàm


In [21]:
def random_tree(rng, n):
    """Cây ngẫu nhiên n nút: gắn từng nút mới vào một chỗ trống ngẫu nhiên."""
    root = TreeNode(rng.randint(1, 100))
    slots = [(root, "L"), (root, "R")]
    for _ in range(n - 1):
        parent, side = slots.pop(rng.randrange(len(slots)))
        child = TreeNode(rng.randint(1, 100))
        if side == "L":
            parent.left = child
        else:
            parent.right = child
        slots += [(child, "L"), (child, "R")]
    return root

rng = random.Random(1302)
for _ in range(500):
    root = random_tree(rng, rng.randint(1, 60))
    want = deepest_brute(root)
    for f in ALL[1:]:
        assert f(root) == want
print("Đối chiếu 500 cây ngẫu nhiên: khớp")

Đối chiếu 500 cây ngẫu nhiên: khớp


In [22]:
# Cây lệch 10^4 nút (độ sâu 9999) — kiểm tra các cách không đệ quy và đệ quy đã nâng limit
chain = TreeNode(1)
cur = chain
for i in range(2, 10001):
    cur.right = TreeNode(i % 100 + 1)
    cur = cur.right
t0 = time.perf_counter()
print("BFS:", Solution().deepestLeavesSum(chain), "| stack DFS:", deepest_dfs_iter(chain),
      "| DFS đệ quy (limit nâng):", SolutionDFS().deepestLeavesSum(chain),
      f"({time.perf_counter() - t0:.3f}s)")
full = build_tree(list(range(1, 10001)))
print("cây đầy đủ 10^4 nút:", Solution().deepestLeavesSum(full), "=", sum(range(8192, 10001)))

BFS: 1 | stack DFS: 1 | DFS đệ quy (limit nâng): 1 (0.004s)
cây đầy đủ 10^4 nút: 16454664 = 16454664


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng **mọi** lá thay vì chỉ lá sâu nhất

In [23]:
def wrong_all_leaves(root):
    if root is None:
        return 0
    if root.left is None and root.right is None:
        return root.val
    return wrong_all_leaves(root.left) + wrong_all_leaves(root.right)   # SAI

r = build_tree(examples[0][0])
print("sai:", wrong_all_leaves(r), "| đúng:", Solution().deepestLeavesSum(r))

sai: 20 | đúng: 15


**Vì sao sai:** Ví dụ 1 có lá `5` ở tầng 2, không phải tầng sâu nhất; cộng vào cho `7 + 5 + 8 = 20`. **Sửa:** phải biết độ sâu — dùng BFS hoặc DFS mang `depth`.

### Lỗi 2 — BFS quên reset tổng cho mỗi tầng

In [24]:
def wrong_no_reset(root):
    q = deque([root])
    level_sum = 0                       # SAI: chỉ khởi tạo một lần
    while q:
        for _ in range(len(q)):
            node = q.popleft()
            level_sum += node.val
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return level_sum

r = build_tree(examples[1][0])
print("sai:", wrong_no_reset(r), "(= tổng cả cây) | đúng:", Solution().deepestLeavesSum(r))

sai: 53 (= tổng cả cây) | đúng: 19


**Vì sao sai:** tổng tích luỹ qua mọi tầng ⇒ ra tổng toàn cây. **Sửa:** đặt `level_sum = 0` **bên trong** `while`, trước vòng `for`.

### Lỗi 3 — DFS một lượt không reset khi gặp tầng sâu hơn

In [25]:
def wrong_dfs(root):
    best = {"d": -1, "s": 0}
    def dfs(node, d):
        if node is None:
            return
        if d > best["d"]:
            best["d"] = d
            best["s"] += node.val      # SAI: phải là gán = node.val
        elif d == best["d"]:
            best["s"] += node.val
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    dfs(root, 0)
    return best["s"]

r = build_tree(examples[0][0])
print("sai:", wrong_dfs(r), "| đúng:", Solution().deepestLeavesSum(r))

sai: 22 | đúng: 15


**Vì sao sai:** giá trị của các tầng nông hơn (`1`, `2`, `4`) vẫn nằm trong tổng ⇒ `1+2+4+7+8 = 22`. **Sửa:** khi `d > max_depth` thì **gán** `total = node.val`.

### Lỗi 4 — Đệ quy quá sâu với cây lệch

Python giới hạn đệ quy; cây lệch `10^4` nút làm DFS đệ quy (không nâng limit) ném `RecursionError`.

In [26]:
def naive_height(node):
    return -1 if node is None else 1 + max(naive_height(node.left), naive_height(node.right))

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    naive_height(chain)
    print("không lỗi?")
except RecursionError as e:
    print("Bắt được RecursionError:", str(e)[:60])
finally:
    sys.setrecursionlimit(old)
print("BFS vẫn chạy được:", Solution().deepestLeavesSum(chain))

Bắt được RecursionError: maximum recursion depth exceeded
BFS vẫn chạy được: 1


**Sửa:** dùng BFS / DFS bằng stack tường minh (cách 3, 5), hoặc `sys.setrecursionlimit(20000)` cho bản đệ quy (cách 2, 4).

### Lỗi 5 — Dựng cây test bằng công thức `2i+1, 2i+2`

Mảng LeetCode **không** ghi con của nút `None`, nên công thức chỉ số kiểu heap dựng sai cây.

In [27]:
def wrong_build(values, i=0):
    if i >= len(values) or values[i] is None:
        return None
    node = TreeNode(values[i])
    node.left = wrong_build(values, 2 * i + 1)     # SAI với mảng LeetCode có None
    node.right = wrong_build(values, 2 * i + 2)
    return node

vals = examples[0][0]
print("dựng sai -> đáp án:", Solution().deepestLeavesSum(wrong_build(vals)), "| dựng đúng ->", Solution().deepestLeavesSum(build_tree(vals)))

dựng sai -> đáp án: 7 | dựng đúng -> 15


**Vì sao sai:** trong Ví dụ 1, theo công thức heap thì `8` (chỉ số 12) là con của chỉ số 5 — một vị trí `null` — nên nút `8` **bị mất**, cây sai hình và đáp án thành 7. **Sửa:** dựng bằng hàng đợi như `build_tree`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có chữ **"tầng / level / depth / row"** ⇒ nghĩ ngay **BFS theo tầng** với `for _ in range(len(q))`.
- **"Sâu nhất / nông nhất"** ⇒ BFS: tầng cuối / tầng đầu tiên có lá.
- Muốn **một lượt DFS** ⇒ mang `depth` theo, giữ `(best_depth, aggregate)` và nhớ **reset** khi có độ sâu tốt hơn.
- Cây tới `10^4` nút trong Python ⇒ cẩn thận đệ quy: ưu tiên lời giải lặp.
- Luôn có sẵn `build_tree` + `show` khi tự test bài cây.

## 15. Biến thể của bài

1. **LeetCode 102 — Binary Tree Level Order Traversal**: trả về danh sách giá trị từng tầng (nền tảng của bài này).
2. **LeetCode 1161 — Maximum Level Sum of a Binary Tree**: tầng có tổng lớn nhất.
3. **LeetCode 513 — Find Bottom Left Tree Value**: nút trái nhất của tầng sâu nhất.
4. **LeetCode 111 — Minimum Depth**: tầng nông nhất có lá ⇒ BFS dừng sớm.
5. **LeetCode 1123 / 865 — Lowest Common Ancestor of Deepest Leaves**: tổ tiên chung thấp nhất của các lá sâu nhất.
6. **Tổng các lá nông nhất** (bài tập 5 bên dưới).

In [28]:
# Biến thể 3: nút trái nhất ở tầng sâu nhất (BFS, phần tử đầu của tầng cuối)
def bottom_left(root):
    q = deque([root])
    first = root.val
    while q:
        first = q[0].val
        for _ in range(len(q)):
            node = q.popleft()
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return first

print("bottom-left ví dụ 1:", bottom_left(build_tree(examples[0][0])))
print("bottom-left ví dụ 2:", bottom_left(build_tree(examples[1][0])))

bottom-left ví dụ 1: 7
bottom-left ví dụ 2: 9


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê đường đi | Lưu mọi đường gốc→lá | `O(n·h)` | `O(n·h)` | Chỉ để kiểm chứng |
| 2. Hai lượt | Tìm chiều cao rồi cộng | `O(n)` | `O(h)` | Dễ nghĩ, theo gợi ý đề |
| 3. BFS theo tầng | Tổng tầng cuối cùng | `O(n)` | `O(w)` | Bản nộp chuẩn, không đệ quy |
| 4. DFS đệ quy 1 lượt | `(max_depth, total)` + reset | `O(n)` | `O(h)` | Cây rộng, code ngắn |
| 5. DFS stack | Như 4, stack tường minh | `O(n)` | `O(h)` | Cây sâu, tránh RecursionError |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().deepestLeavesSum(build_tree([1, 2, 3]))` = ? Còn `build_tree([1, 2, 3, None, 4, 5])`?

**Bài 2 (tính tay):** Vẽ cây `[1,2,3,4,None,None,5,6]` và tính tổng lá sâu nhất.

**Bài 3 (điền TODO):** Hoàn thành `level_values(root)` trả về list các list giá trị từng tầng (LeetCode 102).

**Bài 4 (tự viết lại):** Viết lại DFS một lượt bằng **stack tường minh** không nhìn code: `my_deepest(root)`.

**Bài 5 (mini challenge):** Viết `shallowest_leaves_sum(root)`: tổng các **lá ở tầng nông nhất có lá** (BFS dừng sớm).

In [29]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def level_values_todo(root):
    res = []
    q = deque([root]) if root else deque()
    while q:
        # TODO: lấy trọn một tầng, append list giá trị vào res
        pass
    return res

def my_deepest_todo(root):
    # TODO: stack [(root, 0)], giữ max_depth và total
    pass

def shallowest_leaves_sum_todo(root):
    # TODO: BFS; tầng đầu tiên có lá -> trả tổng các lá trong tầng đó
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `[1,2,3]`: tầng 1 gồm `2, 3`. (2) Cây thứ hai: `2` có con phải `4`, `3` có con trái `5`. (3) Tầng sâu nhất là `4, 5`.

**Bài 2:** (1) Dựng theo hàng đợi: `1 → (2, 3)`, `2 → (4, None)`, `3 → (None, 5)`, `4 → (6, …)`. (2) Tầng 3 chỉ có `6`. (3) Đáp án `6`.

**Bài 3:** (1) `size = len(q)`. (2) Lặp `size` lần, `popleft`, gom giá trị. (3) Đẩy con trái rồi con phải.

**Bài 4:** (1) Stack chứa cặp `(node, depth)`. (2) Ba trường hợp `>`, `==`, `<`. (3) Chỉ đẩy con khác `None`.

**Bài 5:** (1) BFS theo tầng. (2) Trong tầng, cộng giá trị các nút là lá. (3) Nếu tổng tầng có ít nhất một lá thì trả về ngay.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1, 2
assert Solution().deepestLeavesSum(build_tree([1, 2, 3])) == 5
assert Solution().deepestLeavesSum(build_tree([1, 2, 3, None, 4, 5])) == 9
show(build_tree([1, 2, 3, 4, None, None, 5, 6]))
assert Solution().deepestLeavesSum(build_tree([1, 2, 3, 4, None, None, 5, 6])) == 6

# Bài 3
def level_values(root):
    res = []
    q = deque([root]) if root else deque()
    while q:
        cur = []
        for _ in range(len(q)):
            node = q.popleft()
            cur.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        res.append(cur)
    return res

assert level_values(build_tree(examples[1][0])) == [[6], [7, 8], [2, 7, 1, 3], [9, 1, 4, 5]]
assert level_values(None) == []
print("Bài 1–3: OK")

        5
    3
1
    2
        4
            6
Bài 1–3: OK


In [31]:
# Bài 4
def my_deepest(root):
    max_depth, total = -1, 0
    stack = [(root, 0)]
    while stack:
        node, d = stack.pop()
        if d > max_depth:
            max_depth, total = d, node.val
        elif d == max_depth:
            total += node.val
        for child in (node.left, node.right):
            if child:
                stack.append((child, d + 1))
    return total

# Bài 5
def shallowest_leaves_sum(root):
    q = deque([root])
    while q:
        s, found = 0, False
        for _ in range(len(q)):
            node = q.popleft()
            if node.left is None and node.right is None:
                s += node.val
                found = True
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        if found:
            return s
    return 0

def shallowest_brute(root):
    leaves = []
    def dfs(node, d):
        if node is None:
            return
        if node.left is None and node.right is None:
            leaves.append((d, node.val))
        dfs(node.left, d + 1)
        dfs(node.right, d + 1)
    dfs(root, 0)
    m = min(d for d, _ in leaves)
    return sum(v for d, v in leaves if d == m)

assert shallowest_leaves_sum(build_tree(examples[0][0])) == 5
assert shallowest_leaves_sum(build_tree(examples[1][0])) == 1
rng = random.Random(21)
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 40))
    assert my_deepest(root) == deepest_brute(root)
    assert shallowest_leaves_sum(root) == shallowest_brute(root)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS theo tầng:**

```python
q = deque([root])
while q:
    level_sum = 0                       # khởi tạo cho tầng mới
    for _ in range(len(q)):             # đúng số nút của tầng
        node = q.popleft()
        level_sum += node.val
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)
return level_sum                        # tầng cuối
```

**Template DFS một lượt "tốt nhất theo độ sâu":**

```python
best_d, agg = -1, 0
stack = [(root, 0)]
while stack:
    node, d = stack.pop()
    if d > best_d:   best_d, agg = d, node.val     # RESET
    elif d == best_d: agg += node.val
    ...  # đẩy con với d + 1
```

**Checklist trước khi submit:**
- [ ] Chỉ cộng nút ở **tầng sâu nhất** (không phải mọi lá).
- [ ] `level_sum` reset **mỗi tầng** / `total` **gán lại** khi gặp tầng sâu hơn.
- [ ] Không đẩy `None` vào hàng đợi/stack (hoặc xử lý nếu có).
- [ ] Cây lệch `10^4` nút: dùng lặp hoặc nâng recursion limit.
- [ ] Đúng chữ ký: `deepestLeavesSum(self, root: TreeNode | None) -> int`.